# DeepTaxa Tutorial
**DeepTaxa** classifies bacterial 16S rRNA gene sequences into 7 taxonomic ranks (Domain → Species) using a hybrid CNN-BERT deep learning model (92.96% species-level accuracy).

This notebook covers:
1. Installation
2. HuggingFace login
3. Downloading the pre-trained model
4. Downloading sample test data (Greengenes2)
5. Running prediction
6. Inspecting results

## Step 1 — Install DeepTaxa

In [ ]:
!git clone https://github.com/systems-genomics-lab/deeptaxa.git
%cd deeptaxa
!pip install -q .
!deeptaxa --version

## Step 2 — Log in to HuggingFace
You need a HuggingFace account and a token with **read** access.
Get your token at: https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login
login()  # Enter your HuggingFace token when prompted

## Step 3 — Download Pre-trained Model
Two models are available:
- `deeptaxa-full-length-v1.pt` — full-length 16S sequences (92.96% species accuracy)
- `deeptaxa-v3v4-v1.pt` — V3/V4 hypervariable region only (87.55% species accuracy)

In [ ]:
import os
os.makedirs('/content/deeptaxa-data/models', exist_ok=True)

from huggingface_hub import hf_hub_download
model_path = hf_hub_download(
    repo_id='systems-genomics-lab/deeptaxa',
    filename='deeptaxa-full-length-v1.pt',
    local_dir='/content/deeptaxa-data/models'
)
print('Model downloaded to:', model_path)

## Step 4 — Inspect the Checkpoint

In [ ]:
!deeptaxa describe \
    --checkpoint /content/deeptaxa-data/models/deeptaxa-full-length-v1.pt

## Step 5 — Download Sample Test Data (Greengenes2 2024.09)

In [ ]:
os.makedirs('/content/deeptaxa-data/greengenes', exist_ok=True)

from huggingface_hub import hf_hub_download
for fname in ['gg_2024_09_testing.fna.gz', 'gg_2024_09_testing.tsv.gz']:
    path = hf_hub_download(
        repo_id='systems-genomics-lab/greengenes',
        filename=fname,
        repo_type='dataset',
        local_dir='/content/deeptaxa-data/greengenes'
    )
    print('Downloaded:', path)

## Step 6 — Preview Input Data

In [ ]:
import gzip

print('=== First 3 FASTA sequences ===')
count = 0
with gzip.open('/content/deeptaxa-data/greengenes/gg_2024_09_testing.fna.gz', 'rt') as f:
    for line in f:
        print(line, end='')
        if line.startswith('>') and count > 0:
            break
        if line.startswith('>'):
            count += 1

print('\n=== First 3 taxonomy labels ===')
import pandas as pd
tax = pd.read_csv('/content/deeptaxa-data/greengenes/gg_2024_09_testing.tsv.gz', sep='\t', nrows=3)
print(tax)

## Step 7 — Run Prediction with Evaluation
Since we have the true taxonomy labels, this will compute accuracy at all 7 ranks.

In [ ]:
os.makedirs('/content/deeptaxa-outputs/predictions', exist_ok=True)

!deeptaxa predict \
    --fasta-file /content/deeptaxa-data/greengenes/gg_2024_09_testing.fna.gz \
    --taxonomy-file /content/deeptaxa-data/greengenes/gg_2024_09_testing.tsv.gz \
    --checkpoint /content/deeptaxa-data/models/deeptaxa-full-length-v1.pt \
    --output-dir /content/deeptaxa-outputs/predictions

## Step 8 — Inspect Results

In [ ]:
import os, json

output_dir = '/content/deeptaxa-outputs/predictions'
print('Output files:')
for f in os.listdir(output_dir):
    print(' ', f)

# Load and display prediction results
pred_files = [f for f in os.listdir(output_dir) if f.endswith('.tsv') or f.endswith('.csv')]
if pred_files:
    df = pd.read_csv(os.path.join(output_dir, pred_files[0]), sep='\t')
    print('\nPrediction sample (first 5 rows):')
    print(df.head())

# Load metrics if available
metric_files = [f for f in os.listdir(output_dir) if f.endswith('.json')]
if metric_files:
    with open(os.path.join(output_dir, metric_files[0])) as f:
        metrics = json.load(f)
    print('\nAccuracy by taxonomic rank:')
    for rank, val in metrics.items():
        if 'accuracy' in rank.lower() or 'acc' in rank.lower():
            print(f'  {rank}: {val:.4f}')

## (Optional) Step 9 — Predict on Your Own Sequences
Upload a FASTA file with your own 16S rRNA sequences and run prediction.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload your .fna or .fna.gz file

fasta_name = list(uploaded.keys())[0]
os.makedirs('/content/deeptaxa-outputs/my_predictions', exist_ok=True)

!deeptaxa predict \
    --fasta-file /content/{fasta_name} \
    --checkpoint /content/deeptaxa-data/models/deeptaxa-full-length-v1.pt \
    --output-dir /content/deeptaxa-outputs/my_predictions

print('\nResults:')
for f in os.listdir('/content/deeptaxa-outputs/my_predictions'):
    print(' ', f)